# 01. PySATL Data API

## Section 1. Orientation

The PySATL-CPD Data API is the layer that turns raw arrays, tables, and labels into a stable vocabulary for the rest of the project. Change point detectors do not want to care whether a signal started as a NumPy array or a pandas `DataFrame`; benchmark code does not want to care how states or transitions were encoded. The data layer is where those concerns are resolved once, in one consistent shape.

This notebook is deliberately detailed because the whole tutorial series leans on the abstractions introduced here. We will start with plain providers, add labels and segment metadata, build labeled providers, explore slicing and querying operations, and finally move up to datasets, transformers, and file loaders. Every major concept is accompanied by executable code, and every subsection closes with a short summary so the notebook can serve both as a first tutorial and as a reference chapter.


## What in this notebook

By the end of this notebook, you should be able to answer four practical questions.

- How do I represent a raw univariate or multivariate time series before labels are available?
- How do I attach ground-truth segment information and recover change points, states, and transitions from it?
- How do I group several labeled providers into one benchmark-ready dataset and derive state-specific or transition-specific subsets from it?
- How do I move between generated data, transformed feature views, and file-backed datasets without leaving the public data-layer model?

The generator notebook will show how to produce synthetic series. This notebook explains what those generated objects become once they enter the common data API used by detectors and benchmarks.


## Interface Overview

Before we touch concrete classes, it helps to see the shape of the data layer as a ladder of abstractions. A `DataProvider` is the smallest building block: it is a sequential container with an annotation, a length, and the ability to cut and merge data. A `LabeledData` provider adds segment-aware behavior: it stores an ordered labeling, derives change points from segment boundaries, and can expose segment or bisegment views.

Above individual providers sits `IDataset`, the shared collection abstraction. A dataset behaves like a sequence of labeled providers, but it also knows how to split, merge, and aggregate state and transition information across the collection. The concrete `Dataset` class is the general-purpose implementation for regular labeled time series collections, while `StateDataset` is the special case for no-change windows associated with one state.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from pysatl_cpd.data import (
    Dataset,
    FolderCsvColumns,
    IDataset,
    NDArrayMultivariateProvider,
    NDArrayUnivariateProvider,
    PandasLabeledData,
    PlainMultivariateLabeledData,
    PlainUnivariateLabeledData,
    StateDataset,
    TimeseriesAnnotation,
    TransitionDescriptor,
    UnlabeledTimeseriesAnnotation,
    load_folder_csv_dataset,
)
from pysatl_cpd.data.providers.labeled.segments_labeling import SegmentsLabeling
from pysatl_cpd.data.providers.plain.pd_provider import PandasDataProvider
from pysatl_cpd.data.providers.transformers.columns import (
    ColumnFeatureCreator,
    ColumnsSelectorTransformer,
)
from pysatl_cpd.data.typedefs import (
    BisegmentAnnotation,
    NoChangeSeriesAnnotation,
    SegmentAnnotation,
    SegmentInfo,
    StateDescriptor,
    frozendict,
)


def _find_repo_root(start=None):
    import pathlib

    cwd = pathlib.Path(start or pathlib.Path.cwd())
    for parent in [cwd] + list(cwd.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return cwd

## Section 2. Providers

Providers solve the first practical problem in the data layer: how do we keep raw observations together with minimal metadata before ground-truth change information exists? The answer matters because later layers should be able to assume a common provider interface even when labels are still absent.

```python
class DataProvider:
    @property
    def annotation(self) -> AnnotationT: ...

    def __len__(self) -> int: ...
    def __iter__(self) -> Iterator[DataT]: ...

    def cut(self, start: int, stop: int, *, annotation=None) -> Self: ...
    @classmethod
    def merge(cls, providers, annotation_builder=None) -> Self: ...
```


In this section, we will look at the three most immediate representations: one-dimensional NumPy data, two-dimensional NumPy data, and pandas-backed tables. The concrete examples are intentionally small so the API shape is easy to read.


### 2.1 NDArrayUnivariateProvider

When a time series has exactly one numeric observation per time step, the simplest useful provider is a one-dimensional NumPy wrapper. The motivation for this class is not just convenience. It gives the rest of the project a uniform interface for iteration, slicing, and annotation handling, while keeping the representation as lightweight as possible.

The key API idea is that `NDArrayUnivariateProvider` behaves like a sequential container. You can iterate through observations, ask for its length, inspect the copied `raw_data`, and cut out an inclusive slice while optionally attaching a fresh annotation.


In [ ]:
univariate_values = np.array([0.5, 0.7, 1.1, 0.9, 1.3], dtype=np.float64)
univariate_provider = NDArrayUnivariateProvider(
    univariate_values,
    UnlabeledTimeseriesAnnotation(name="univariate_demo"),
)

print("Annotation:", univariate_provider.annotation)
print("Length:", len(univariate_provider))
print("Iterated values:", list(univariate_provider))
print("Raw data copy:", univariate_provider.raw_data)

univariate_slice = univariate_provider.cut(
    1,
    3,
    annotation=UnlabeledTimeseriesAnnotation(name="univariate_demo_slice"),
)
print("Slice values:", list(univariate_slice))
print("Slice annotation:", univariate_slice.annotation)

### 2.2 NDArrayMultivariateProvider

The next problem is multivariate data. A detector or benchmark often wants several numeric features at each time step, but it still wants the same provider contract: iterate step-by-step, preserve annotation metadata, and allow consistent slicing. A plain two-dimensional array is a natural storage choice, but it still needs to be wrapped into the provider model.

`NDArrayMultivariateProvider` treats rows as time steps and columns as features. Iteration yields one feature row at a time, while `raw_data` gives a copied matrix for inspection, plotting, or comparison with transformed versions later in the notebook.


In [ ]:
multivariate_values = np.array(
    [
        [0.5, 10.0],
        [0.7, 10.2],
        [1.1, 10.4],
        [0.9, 10.7],
    ],
    dtype=np.float64,
)
multivariate_provider = NDArrayMultivariateProvider(
    multivariate_values,
    UnlabeledTimeseriesAnnotation(name="multivariate_demo"),
)

print("Annotation:", multivariate_provider.annotation)
print("Length:", len(multivariate_provider))
print("Rows from iteration:", [row.tolist() for row in multivariate_provider])
print("Raw data shape:", multivariate_provider.raw_data.shape)

multivariate_slice = multivariate_provider.cut(
    1,
    2,
    annotation=UnlabeledTimeseriesAnnotation(name="multivariate_demo_slice"),
)
print("Slice rows:", [row.tolist() for row in multivariate_slice])

### 2.3 PandasDataProvider

A raw NumPy matrix is compact, but many workflows are easier when features have names. That is the problem the pandas-backed provider solves. It keeps the same provider semantics while preserving column labels, which later becomes important for feature selection, plotting, and benchmark-time transformations.

The important API distinction is that a `PandasDataProvider` still behaves like a provider first and a table second. You can iterate through it like any other provider, but you can also ask for the copied `dataset` and the list of `columns` to stay in a more tabular mindset.
Beyond basic iteration, `select_columns()` returns a new `PandasDataProvider` with only the named columns. The optional `rename_provider=True` flag appends the selected column names to the annotation name, which makes the result easier to identify later in a workflow.


In [ ]:
pandas_frame = pd.DataFrame(
    {
        "value": [0.5, 0.7, 1.1, 0.9],
        "aux": [10.0, 10.2, 10.4, 10.7],
    }
)
pandas_provider = PandasDataProvider(
    pandas_frame,
    UnlabeledTimeseriesAnnotation(name="pandas_demo"),
)

print("Columns:", list(pandas_provider.columns))
print("Length:", len(pandas_provider))
print("Rows from iteration:", [row.tolist() if hasattr(row, "tolist") else row for row in pandas_provider])
display(pandas_provider.dataset)

selected_plain = pandas_provider.select_columns(["value"], rename_provider=True)
print("Selected annotation:", selected_plain.annotation)
display(selected_plain.dataset)

**Summary of Section 2:** Unlabeled providers all solve the same interface problem, but they do it with different storage choices. Use the univariate NumPy provider for single-feature scalar signals, the multivariate NumPy provider for matrix-shaped numeric data, and the pandas provider when named columns make later operations easier to read and safer to configure.


## Section 3. Labels, Segments, and Ground Truth

A raw provider tells us what values were observed, but it does not tell us where the known change points are or what each regime means. This section solves that second problem by introducing the segment vocabulary that all labeled providers share.

The central idea is simple: a labeled time series is built from an ordered sequence of segment descriptors. Once those descriptors exist, change points, states, transitions, segment slices, and transition slices all become derived views of one source of truth.


### 3.1 StateDescriptor

Before we describe a segment, we need a language for its state. A change-point benchmark rarely cares only that "a segment exists"; it usually cares whether the segment is baseline, shifted, anomalous, noisy, or part of some named regime family. The state descriptor is the lightweight key-value vocabulary used for that purpose.

API-wise, `StateDescriptor` is intentionally simple. It behaves like an immutable mapping of small state attributes, so it can be compared, stored in sets, reused in filters, and printed in a compact way. The notebook series will keep reusing this same object when it talks about dataset states, transitions, and state-specific ARL slices.


In [ ]:
baseline_state = StateDescriptor(type="baseline", regime="stable")
shifted_state = StateDescriptor(type="shifted", regime="changed")

print("Baseline state:", baseline_state)
print("Baseline as dict:", dict(baseline_state))
print("Shifted state:", shifted_state)
print("States compare by content:", baseline_state == StateDescriptor(type="baseline", regime="stable"))

### 3.2 SegmentInfo

Once a state vocabulary exists, the next problem is describing one concrete segment in one concrete time series. A segment needs boundaries, an order in the sequence, and a state label. That is exactly what `SegmentInfo` provides.

The key API fields are `segment_num`, `segment_start`, `segment_end`, and `state`. Because change points in this project are zero-based, the segment boundaries are also expressed in zero-based coordinates. Later code never has to guess where a segment begins or ends; it reads those boundaries directly from the segment table.


In [ ]:
segment_rows = [
    SegmentInfo(segment_num=0, segment_start=0, segment_end=5, state=baseline_state),
    SegmentInfo(segment_num=1, segment_start=6, segment_end=10, state=shifted_state),
    SegmentInfo(segment_num=2, segment_start=11, segment_end=14, state=baseline_state),
]

segment_table = pd.DataFrame(
    {
        "segment_num": [segment.segment_num for segment in segment_rows],
        "segment_start": [segment.segment_start for segment in segment_rows],
        "segment_end": [segment.segment_end for segment in segment_rows],
        "state": [dict(segment.state) for segment in segment_rows],
    }
)
display(segment_table)

### 3.3 Change Points, States, and Transitions

The main motivation for storing segment rows instead of only raw change-point indices is that richer structure can be derived from them automatically. Once segments are ordered, change points are just the starts of later segments. States are the unique segment states. Transitions are the pairs of states that appear between adjacent segments.

For the API user, this means you do not manually maintain three separate sources of truth. You define segments once, and labeled providers expose `change_points`, `states`, and `transitions` as derived properties.


In [ ]:
manual_series = np.array([0.2, -0.1, 0.3, 0.1, 0.0, 0.2, 3.0, 3.3, 2.9, 3.1, 2.8, 0.1, -0.2, 0.0, 0.2])
# Change points are the start indices of all segments except the first
derived_change_points = tuple(segment.segment_start for segment in segment_rows[1:])
# States are the unique state descriptors across segments
derived_states = {segment.state for segment in segment_rows}
# Transitions are adjacent pairs between consecutive segments
derived_transitions = [
    TransitionDescriptor(curr_state=segment_rows[i].state, next_state=segment_rows[i + 1].state)
    for i in range(len(segment_rows) - 1)
]
print("Zero-based change points:", derived_change_points)
print("States:", list(derived_states))
print("Transitions:")
for tr in derived_transitions:
    print(tr)

Labeled providers store their segment table as a `SegmentsLabeling` — a `Sequence[SegmentInfo]` that also provides query and transformation methods. You will see it in action as the `.segments_labeling` property once we build labeled providers in §4.


### 3.4 Annotations
Every provider carries an annotation that describes what the data represents. PySATL defines a small hierarchy of annotation types, each tuned for a different stage of the data lifecycle.


#### 3.4.1 TimeseriesAnnotation
`TimeseriesAnnotation` is the base annotation for all labeled time series. It stores a `name`, an optional `source` (e.g. a file path or generator name), and optional immutable `metadata`. The more specific annotation types below extend it.


In [ ]:
ts_ann = TimeseriesAnnotation(
    name="demo_series",
    source="notebook",
    metadata=frozendict(version="1.0"),
)
print("Name:", ts_ann.name)
print("Source:", ts_ann.source)
print("Type:", type(ts_ann).__name__)

#### 3.4.2 UnlabeledTimeseriesAnnotation
`UnlabeledTimeseriesAnnotation` extends `TimeseriesAnnotation` for data without segment labels. We already used it in §2 with raw providers. Its presence makes the lack of labels explicit while keeping the same base fields.


In [ ]:
unlabeled_ann = UnlabeledTimeseriesAnnotation(name="raw_series")
print("Name:", unlabeled_ann.name)
print("Type:", type(unlabeled_ann).__name__)

#### 3.4.3 SegmentAnnotation
`SegmentAnnotation` extends `TimeseriesAnnotation` with a `state` field. This is the annotation attached to providers returned by `query_segments()` — it records which regime the segment belongs to.


In [ ]:
seg_ann = SegmentAnnotation(
    name="baseline_segment",
    state=baseline_state,
)
print("Name:", seg_ann.name)
print("State:", dict(seg_ann.state))
print("Type:", type(seg_ann).__name__)

#### 3.4.4 BisegmentAnnotation and TransitionDescriptor
`BisegmentAnnotation` extends `TimeseriesAnnotation` with a `transition` field. Its type is `TransitionDescriptor`, a frozen dataclass with `curr_state` and `next_state`. This is what `query_bisegments()` attaches — it names both sides of a change point.


In [ ]:
transition = TransitionDescriptor(
    curr_state=baseline_state,
    next_state=shifted_state,
)
biseg_ann = BisegmentAnnotation(
    name="baseline_to_shifted",
    transition=transition,
)
print("Name:", biseg_ann.name)
print("Current state:", dict(biseg_ann.transition.curr_state))
print("Next state:", dict(biseg_ann.transition.next_state))
print("Type:", type(biseg_ann).__name__)

#### 3.4.5 NoChangeSeriesAnnotation
`NoChangeSeriesAnnotation` extends `SegmentAnnotation` for a series that contains no change points. It carries a `state` just like `SegmentAnnotation`, but signals that the entire span belongs to one regime. This is used by `StateDataset` to mark no-change windows.


In [ ]:
nochange_ann = NoChangeSeriesAnnotation(
    name="stable_run",
    state=baseline_state,
)
print("Name:", nochange_ann.name)
print("State:", dict(nochange_ann.state))
print("Type:", type(nochange_ann).__name__)

**Summary of Section 3:** `StateDescriptor` explains what regime a segment belongs to, `SegmentInfo` explains where that regime lives in the series, and everything else that matters for ground truth flows from those segment rows. Change points are zero-based boundaries derived from the same labeling that later powers segment and bisegment queries.
The annotation hierarchy introduced in §3.4 mirrors the lifecycle of data: `TimeseriesAnnotation` (base) → `UnlabeledTimeseriesAnnotation` (pre-label data), `SegmentAnnotation` (single segment with `state`), `BisegmentAnnotation` (transition window with `transition`), and `NoChangeSeriesAnnotation` (stable no-change series).


### 3.5 SegmentsLabeling

`SegmentsLabeling` is the container that wraps the ordered list of `SegmentInfo` into a validated, queryable sequence. Every labeled provider stores its segment information as a `SegmentsLabeling` internally — it is the glue between the `SegmentInfo` rows built in §3.2–§3.3 and the concrete labeled provider constructors in §4.

The class validates that segments are contiguous and non-overlapping. Beyond validation, it powers the `query_segments()` and `query_bisegments()` methods that produce collections of segment views. Calling `provider.query_segments(filter_fn)` returns a list of labeled providers, each trimmed to one matching segment with its own annotation and change points. This makes per-segment analysis natural — iterate over segments, compute statistics per regime, or feed individual segment windows into a detector. Similarly, `query_bisegments()` returns two-segment windows centered on each change point, the natural unit for transition-aware evaluation.

You rarely construct a `SegmentsLabeling` by hand in day-to-day use — the labeled provider factories do it automatically. But understanding it explains why the `segment_info` parameter of `from_unlabeled_data()` expects an iterable of `SegmentInfo`, what validation it undergoes, and how the same segment row indexing powers the query operations in §5.

The example below shows direct construction from the `segment_rows` we built in §3.2 and a tabular view via `to_dataframe()`, which includes both the boundary columns and the state descriptor keys (e.g. `type`, `regime`) as columns.


In [ ]:
segments_labeling = SegmentsLabeling(segment_rows)
print("Number of segments:", len(segments_labeling))
print("Segment states:", [dict(s) for s in segments_labeling.states])
print("Data length:", segments_labeling.data_len)
display(segments_labeling.to_dataframe())

## Section 4. Labeled Providers

Now we can solve the real practical problem: how do we combine raw provider data with ordered segment labeling and keep the result easy to slice, inspect, and reuse? The answer is the family of labeled providers.

The same logical model appears in several concrete implementations. The plain univariate and multivariate variants are NumPy-backed. The pandas-backed variant keeps named columns and integrates naturally with column selection and tabular inspection.


### 4.1 PlainUnivariateLabeledData

The plain univariate labeled provider is the most compact complete labeled object in the data layer. It is useful when you want a scalar signal with explicit ground truth and you do not need named columns.

Its API combines the raw-provider behavior with labeled-provider behavior. You can iterate through observations, access `raw_data`, inspect `change_points`, and query states or transitions from the same object.
The `from_unlabeled_data()` class method takes an unlabeled provider, a sequence of `SegmentInfo`, and a `TimeseriesAnnotation`, and returns a fully connected labeled provider whose change points, states, and transitions are derived automatically. The `.segments_labeling` property exposes the underlying `SegmentsLabeling` container we discussed in §3.5.


In [ ]:
manual_series = np.array([0.2, -0.1, 0.3, 0.1, 0.0, 0.2, 3.0, 3.3, 2.9, 3.1, 2.8, 0.1, -0.2, 0.0, 0.2])
manual_labeled = PlainUnivariateLabeledData.from_unlabeled_data(
    NDArrayUnivariateProvider(manual_series, UnlabeledTimeseriesAnnotation(name="manual_labeled_univariate")),
    segment_rows,
    TimeseriesAnnotation(name="manual_labeled_univariate"),
)
print("Annotation:", manual_labeled.annotation)
print("Length:", len(manual_labeled))
print("Raw data:", manual_labeled.raw_data)
print("Change points:", list(manual_labeled.change_points))
print("States:", [dict(state) for state in manual_labeled.states])
# The SegmentsLabeling container
labeling = manual_labeled.segments_labeling
print(f"\nSegmentsLabeling: {len(labeling)} segments, type={type(labeling).__name__}")
for seg in labeling:
    print(f"  Segment {seg.segment_num}: {seg.segment_start}--{seg.segment_end} state={dict(seg.state)}")

### 4.2 PlainMultivariateLabeledData

A multivariate labeled provider solves the same problem, but now each time step is a feature row rather than a scalar. This is the most natural format for plain synthetic multivariate data when you want to stay NumPy-backed.

The API difference is small but important: `raw_data` is now a matrix, and iteration yields rows. All higher-level label-derived behavior, including change points and segment queries, stays conceptually the same.


In [ ]:
multivariate_labeling = [
    SegmentInfo(segment_num=0, segment_start=0, segment_end=2, state=baseline_state),
    SegmentInfo(segment_num=1, segment_start=3, segment_end=4, state=shifted_state),
]
multivariate_labeled = PlainMultivariateLabeledData.from_unlabeled_data(
    NDArrayMultivariateProvider(
        np.array([[0.0, 10.0], [0.2, 10.2], [-0.1, 9.8], [3.0, 20.0], [2.8, 19.7]], dtype=np.float64),
        UnlabeledTimeseriesAnnotation(name="manual_labeled_multivariate"),
    ),
    multivariate_labeling,
    TimeseriesAnnotation(name="manual_labeled_multivariate"),
)

print("Raw shape:", multivariate_labeled.raw_data.shape)
print("Rows:", [row.tolist() for row in multivariate_labeled])
print("Change points:", list(multivariate_labeled.change_points))

### 4.3 PandasLabeledData

Pandas-backed labeled data is the richest form for day-to-day work because it preserves feature names. That makes it easier to inspect the data table, to select columns, and later to configure benchmarks that operate on specific feature subsets.

The most important methods are `feature_columns`, `dataset()`, `select_columns()`, and `create_feature_column()`. They let you stay in one labeled-provider abstraction while still doing practical table-shaped manipulations.

The example below constructs a `PandasLabeledData` from a manually created DataFrame with two feature columns (`value` and `aux`) and a three-segment baseline–shift–baseline labeling.


In [ ]:
pandas_data = pd.DataFrame(
    {
        "value": [0.2, -0.3, 0.5, -0.1, 0.4, 0.1, 3.1, 2.9, 3.3, 2.8, 3.0, 0.3, -0.2, 0.0, 0.1],
        "aux": [5.1, 4.9, 5.2, 5.0, 4.8, 5.1, 7.0, 7.2, 6.9, 7.1, 6.8, 5.0, 5.2, 4.9, 5.1],
    }
)
pandas_labeled = PandasLabeledData.from_unlabeled_data(
    PandasDataProvider(pandas_data, UnlabeledTimeseriesAnnotation(name="pandas_labeled_demo")),
    [
        SegmentInfo(segment_num=0, segment_start=0, segment_end=5, state=baseline_state),
        SegmentInfo(segment_num=1, segment_start=6, segment_end=10, state=shifted_state),
        SegmentInfo(segment_num=2, segment_start=11, segment_end=14, state=baseline_state),
    ],
    TimeseriesAnnotation(name="pandas_labeled_demo"),
)
print("Feature columns:", list(pandas_labeled.feature_columns))
print("Change points:", list(pandas_labeled.change_points))
display(pandas_labeled.dataset().head(8))

**Summary of Section 4:** Labeled providers are where raw observations and segment metadata finally become one object. The plain variants keep things lightweight, while the pandas-backed variant adds named columns and richer table operations. Builder helpers let generated series enter this model directly.


## Section 5. Provider Operations

Once data is labeled, the next question is not "what is it?" but "how do I extract the exact view I need?" This section solves that problem. We will slice a labeled provider directly, extract per-segment views, and extract per-transition bisegment views.

These operations matter because later notebooks do not invent new data abstractions for training, inspection, or benchmarking. They reuse these same provider-level operations.


### 5.1 cut()

The motivation for `cut()` is local inspection. Sometimes you want one contiguous window of a labeled series and you want the labels re-based to that window automatically. Doing this manually is error-prone because you would otherwise need to rewrite change-point positions and segment boundaries yourself.

The `cut()` API takes inclusive start and end indices and returns a new labeled provider of the same concrete type. The sliced provider gets adjusted labeling and either a default or user-supplied annotation.


In [ ]:
provider_cut = pandas_labeled.cut(2, 10)
print("Cut annotation:", provider_cut.annotation)
print("Cut length:", len(provider_cut))
print("Cut change points:", list(provider_cut.change_points))
display(provider_cut.dataset())

### 5.2 query_segments()

The problem solved by `query_segments()` is selective state extraction. Instead of cutting raw index ranges by hand, you ask the provider for all segment-level views matching a predicate. That keeps the extraction logic tied to segment semantics rather than to memorized boundaries.

Each returned provider is annotated as a `SegmentAnnotation` (introduced in §3.4.3). That annotation carries the segment's `state`, making the result self-describing even after it has been separated from the original series.


In [ ]:
baseline_segments = pandas_labeled.query_segments(lambda segment: segment.state["type"] == "baseline")
print("Number of baseline segments:", len(baseline_segments))
print("First baseline annotation:", baseline_segments[0].annotation)
print("First baseline change points:", list(baseline_segments[0].change_points))
display(baseline_segments[0].dataset())

### 5.3 query_bisegments()

Change-point evaluation often cares about local transition windows rather than about full time series. That is the motivation for `query_bisegments()`. A bisegment is the two-segment view centered on one true change point, and it is the natural unit for transition-aware no-reset evaluation.

Each returned provider is annotated as a `BisegmentAnnotation` (introduced in §3.4.4). Its `transition` field is a `TransitionDescriptor` containing `curr_state` and `next_state`, naming both sides of the change point.


In [ ]:
bisegments = pandas_labeled.query_bisegments()
first_bisegment = bisegments[0]

print("Number of bisegments:", len(bisegments))
print("First bisegment annotation:", first_bisegment.annotation)
print("First bisegment change points:", list(first_bisegment.change_points))
display(first_bisegment.dataset(state_columns={"state": "regime"}))

### 5.4 merge()

Sometimes the problem is the opposite of slicing: you have several compatible providers and want to stitch them back together into one longer labeled provider. That is exactly what `merge()` solves.

The merge API is defined on the provider type itself. It validates compatibility, merges the raw unlabeled providers, merges segment labeling, and builds a merged annotation. This is the mechanism later reused by dataset-level `merge()` as well.


In [ ]:
left = pandas_labeled.cut(0, 4)
right = pandas_labeled.cut(5, len(pandas_labeled) - 1)
merged_provider = type(pandas_labeled).merge([left, right])

print("Merged length:", len(merged_provider))
print("Merged change points:", list(merged_provider.change_points))
print("Matches original length:", len(merged_provider) == len(pandas_labeled))

**Summary of Section 5:** Provider operations let you move between whole-series, segment-level, and transition-level views without leaving the labeled-provider abstraction. `cut()` is for contiguous windows, `query_segments()` is for state-aware extraction, `query_bisegments()` is for transition-aware extraction, and `merge()` reverses the process when compatible views need to be stitched back together.


## Section 6. Transformers
The next problem is feature preparation. A labeled provider may already have the right labels and segmentation, but not the right feature view for a detector. This is common in multivariate data, where a benchmark or algorithm often wants one column or a small feature subset.
Transformers solve that problem while staying inside the data-provider model. In this section we cover three transformer types: `ColumnsSelectorTransformer` for selecting feature columns, `ColumnFeatureCreator` for deriving new columns, and `ComposedTransformer` for chaining them together.


### 6.1 ColumnsSelectorTransformer

The motivation for `ColumnsSelectorTransformer` is simple: feature selection should be a first-class operation instead of ad hoc `DataFrame` slicing sprinkled through detector code. By packaging the selection as a transformer, you can inspect it, reuse it, and later pass the same object into benchmark entries.

The API takes one column name or a sequence of column names. Its `annotation` property tells you how the transformation is described, and `transform()` applies the feature selection to a compatible pandas-backed labeled provider.


In [ ]:
column_transformer = ColumnsSelectorTransformer(columns=["value"])
selected_provider = column_transformer.transform(pandas_labeled)

print("Transformer annotation:", column_transformer.annotation)
print("Selected columns:", list(selected_provider.feature_columns))
print("Selected provider annotation:", selected_provider.annotation)
display(selected_provider.dataset().head())

### 6.2 ColumnFeatureCreator
`ColumnFeatureCreator` appends a derived column computed row-wise from existing features. Use it when you need a transformation like a ratio, difference, or summary statistic that becomes a new feature column.
The constructor takes the new column name and a callable that receives each pandas row and returns a value. The optional `rename_provider=True` flag updates the provider annotation name to reflect the added column.


In [ ]:
creator = ColumnFeatureCreator(
    name="value_sq",
    mapping=lambda row: row["value"] ** 2,
    rename_provider=True,
)
enhanced = creator.transform(pandas_labeled)
print("Feature columns:", list(enhanced.feature_columns))
print("Provider name:", enhanced.annotation.name)
display(enhanced.dataset().head(3))

### 6.3 ComposedTransformer
`ComposedTransformer` chains multiple transformations into one reusable step using the `&` operator: the left transformer runs first, and its result becomes the input to the right transformer. This keeps the pipeline order visible and avoids temporary variables.
The example below first selects a subset of columns, then appends a derived feature column computed from the remaining features.


In [ ]:
pipeline = ColumnsSelectorTransformer(columns=["value", "aux"]) & ColumnFeatureCreator(
    name="product",
    mapping=lambda row: row["value"] * row["aux"],
)
pipeline_result = pipeline.transform(pandas_labeled)
print("Pipeline annotation:", pipeline.annotation)
print("Output columns:", list(pipeline_result.feature_columns))
print("Change points:", list(pipeline_result.change_points))
display(pipeline_result.dataset().head(4))

### 6.4 Why transformations belong in the data layer

The same feature-selection problem appears in interactive inspection, in detector configuration, and in benchmarking. Keeping the transformation as a data-layer object means all three workflows can share one explicit transformation step instead of reimplementing the same slicing logic.

In practice, the tutorial series will use transformers in two ways. Sometimes we apply them directly for inspection, as we did above. Later, benchmark entries will accept the same transformer so the feature view is part of the detector configuration rather than hidden in notebook-local preprocessing code.


In [ ]:
selected_aux_provider = ColumnsSelectorTransformer(columns=["aux"], rename_provider=True).transform(pandas_labeled)
print("Renamed selected annotation:", selected_aux_provider.annotation)
display(selected_aux_provider.dataset().head())

**Summary of Section 6:** Transformers keep feature preparation explicit and reusable. `ColumnsSelectorTransformer` selects feature columns, `ColumnFeatureCreator` appends derived columns, and `ComposedTransformer` chains steps together with the `&` operator. This becomes especially important once detectors and benchmarks start operating on multivariate labeled data.


## Section 7. Datasets

Individual labeled providers are useful for local inspection, but experiments and benchmarks normally work with collections. This section addresses that collection-level problem. We will create datasets, inspect their shared state and transition information, split them, filter them, merge them, and derive fixed-state no-change windows.

The most important idea is that datasets do not invent a new data model. They are organized collections of the same labeled providers you have already seen.


### 7.1 Creating a Dataset

A `Dataset` is the general-purpose collection type for labeled providers. The motivation for this class is not just grouping objects together. It adds collection-level semantics such as the union of states and transitions, train/test splitting, and dataset-wide filtering.

The API behaves like a sequence, so you can index or iterate over it directly. On top of that, properties such as `states` and `transitions` summarize the collection as a whole.


In [ ]:
dataset = Dataset(
    [
        PlainUnivariateLabeledData.from_unlabeled_data(
            NDArrayUnivariateProvider(
                np.array([0.1, 0.3, -0.2, 0.4], dtype=np.float64),
                UnlabeledTimeseriesAnnotation(name="series_01"),
            ),
            [
                SegmentInfo(segment_num=0, segment_start=0, segment_end=1, state=baseline_state),
                SegmentInfo(segment_num=1, segment_start=2, segment_end=3, state=shifted_state),
            ],
            TimeseriesAnnotation(name="series_01"),
        ),
        PlainUnivariateLabeledData.from_unlabeled_data(
            NDArrayUnivariateProvider(
                np.array([-0.1, 2.9, 3.1, 2.8], dtype=np.float64),
                UnlabeledTimeseriesAnnotation(name="series_02"),
            ),
            [
                SegmentInfo(segment_num=0, segment_start=0, segment_end=0, state=baseline_state),
                SegmentInfo(segment_num=1, segment_start=1, segment_end=3, state=shifted_state),
            ],
            TimeseriesAnnotation(name="series_02"),
        ),
        PlainUnivariateLabeledData.from_unlabeled_data(
            NDArrayUnivariateProvider(
                np.array([0.5, 0.0, -0.3, 0.2, -0.1], dtype=np.float64),
                UnlabeledTimeseriesAnnotation(name="series_03"),
            ),
            [
                SegmentInfo(segment_num=0, segment_start=0, segment_end=2, state=baseline_state),
                SegmentInfo(segment_num=1, segment_start=3, segment_end=4, state=shifted_state),
            ],
            TimeseriesAnnotation(name="series_03"),
        ),
        PlainUnivariateLabeledData.from_unlabeled_data(
            NDArrayUnivariateProvider(
                np.array([2.9, 3.2, 3.0, 0.2, -0.1, 0.3], dtype=np.float64),
                UnlabeledTimeseriesAnnotation(name="series_04"),
            ),
            [
                SegmentInfo(segment_num=0, segment_start=0, segment_end=2, state=shifted_state),
                SegmentInfo(segment_num=1, segment_start=3, segment_end=5, state=baseline_state),
            ],
            TimeseriesAnnotation(name="series_04"),
        ),
    ]
)
print("Dataset type via public interface:", isinstance(dataset, IDataset))
print("Dataset size:", len(dataset))
print("Provider names:", [provider.annotation.name for provider in dataset])
print("Dataset states:", [dict(state) for state in dataset.states])
print("Dataset transitions:", [repr(transition) for transition in dataset.transitions])

### 7.2 filter_by_annotation(), filter_by_segments(), and filter_by_bisegments()

Once data is grouped into a dataset, the next problem is building focused subsets. Sometimes you filter by whole-provider annotation, sometimes by segment-level state, and sometimes by transition-level bisegment criteria. The dataset filtering helpers solve these three related but distinct collection problems.

The APIs return new datasets whose providers are already in the right form for the requested view. That means segment filtering returns a dataset of segment providers, and bisegment filtering returns a dataset of bisegment providers, rather than merely returning raw indices.


In [ ]:
annotation_filtered = dataset.filter_by_annotation(
    lambda annotation: annotation.name.endswith("1") or annotation.name.endswith("2")
)
segment_filtered = dataset.filter_by_segments(lambda segment: segment.state["type"] == "baseline")
bisegment_filtered = dataset.filter_by_bisegments()

print("Annotation-filtered size:", len(annotation_filtered))
print("Segment-filtered size:", len(segment_filtered))
print("Bisegment-filtered size:", len(bisegment_filtered))
print("First bisegment annotation from dataset filter:", bisegment_filtered[0].annotation)

### 7.3 train_test_split() and merge()

A benchmark-ready collection often needs two opposite operations: splitting into subsets and collapsing back into one provider. `train_test_split()` solves the first problem, and dataset-level `merge()` solves the second.

The split API preserves the concrete dataset type, while `merge()` returns one labeled provider containing the data and labeling from all providers in sequence. Together, these two operations make it easy to move between collection-level and provider-level views.


In [ ]:
train_dataset, test_dataset = dataset.train_test_split(test_size=0.25, random_state=13)
merged_dataset_provider = dataset.merge()

print("Train/Test sizes:", len(train_dataset), len(test_dataset))
print("Merged provider length:", len(merged_dataset_provider))
print("Merged provider change points count:", len(merged_dataset_provider.change_points))

### 7.4 StateDataset

Some benchmark scenarios care specifically about no-change windows from one state. That is a different problem from general labeled-series collections, so the project provides `StateDataset` as a specialized abstraction.

`StateDataset.from_dataset()` filters the source dataset by state, merges matching segments, slices them into fixed windows, and annotates the result as no-change providers associated with one state. This is a crucial bridge into later ARL-style evaluation workflows.
The `keep_remainder=True` parameter includes a final shorter window when the segment length is not evenly divisible by `slice_length`; without it, any remainder is discarded.


In [ ]:
state_dataset = StateDataset.from_dataset(
    dataset,
    slice_length=4,
    state=next(iter(dataset.states)),
    keep_remainder=True,
)

print("StateDataset size:", len(state_dataset))
print("StateDataset state:", state_dataset.state)
print("First state-window annotation:", state_dataset[0].annotation)

**Summary of Section 7:** Datasets are the benchmark-facing collection abstraction of the data layer. They expose shared state and transition information, support focused filtering, allow reproducible splitting, and can derive specialized no-change collections through `StateDataset`.


## Section 8. Loading Data From Files

The last practical problem in the data layer is moving from a filesystem layout into the same in-memory dataset abstractions used everywhere else. That loader path is important because real projects rarely start from hard-coded arrays.

The current public loader story centers on segmented CSV folders. The example uses a small pre-created dataset in `assets/userguide/examples/csv_dataset/` so you can inspect the file layout, the required columns, and the resulting labeled provider structure.


### 8.1 FolderCsvColumns and load_folder_csv_dataset()

The loader needs two kinds of information: where to read the files from, and how to interpret their columns. `FolderCsvColumns` provides the column contract, and `load_folder_csv_dataset()` applies that contract to one root directory.

The API assumes one or more subfolders under the root, each with a `metadata.yaml` file and one or more CSV files. Each CSV must include feature columns, state columns, and a segment-number column that identifies contiguous labeled regions.


In [ ]:
ROOT = _find_repo_root()
loaded_dataset = load_folder_csv_dataset(
    ROOT / "assets/userguide/examples/csv_dataset",
    FolderCsvColumns(
        feature_columns=["value", "aux"],
        state_columns=["state_type", "state_regime"],
        segment_num_column="segment_num",
    ),
)

loaded_provider = loaded_dataset[0]
print("Loaded dataset size:", len(loaded_dataset))
print("Loaded provider annotation:", loaded_provider.annotation)
print("Loaded provider change points:", list(loaded_provider.change_points))
display(loaded_provider.dataset())

### 8.2 Why the loaded result matches the rest of the notebook

A loader is only useful if it returns objects that can immediately participate in the same workflows as generated or manually constructed data. The loaded dataset above is not a special-case object; it is an ordinary `Dataset` of labeled providers.

That means the same operations still apply. You can inspect states and transitions, filter by segments or bisegments, apply transformers, and pass the result into online detectors or benchmarks in later notebooks.


In [ ]:
print("Loaded dataset states:", [dict(state) for state in loaded_dataset.states])
print("Loaded dataset transitions:", [repr(transition) for transition in loaded_dataset.transitions])
loaded_bisegments = loaded_dataset.filter_by_bisegments()
print("Loaded dataset bisegment count:", len(loaded_bisegments))

**Summary of Section 8:** The folder CSV loader turns on-disk segmented data into the same in-memory labeled-provider and dataset abstractions used throughout the project. Once loaded, file-backed data behaves just like generated or manually constructed data.


## Section 9. End-to-End Mini Workflow

To close the notebook, we will solve the full local problem once: start from a generated labeled provider, build a dataset, extract a transition-focused subset, and prepare a single-feature view. The goal is not to introduce new APIs, but to show how the main pieces already fit together.


### 9.1 From generated series to benchmark-ready views

The notebook has introduced many small building blocks, and the final challenge is to connect them without losing readability. A realistic data-preparation workflow does not use every class at once, but it does usually cross several abstraction boundaries in sequence.

This short example goes from generation to labeled provider, from labeled provider to dataset, from dataset to bisegment subset, and from multivariate view to single-column view. That is the shape many later experiments will follow.


In [ ]:
workflow_data = pd.DataFrame(
    {
        "value": [0.2, -0.1, 0.3, 0.0, 3.0, 2.9, 3.1, 0.1, -0.2, 0.2],
        "aux": [5.1, 4.9, 5.0, 5.2, 7.0, 7.2, 6.9, 5.0, 5.1, 4.8],
    }
)
workflow_provider = PandasLabeledData.from_unlabeled_data(
    PandasDataProvider(workflow_data, UnlabeledTimeseriesAnnotation(name="workflow_series")),
    [
        SegmentInfo(segment_num=0, segment_start=0, segment_end=3, state=baseline_state),
        SegmentInfo(segment_num=1, segment_start=4, segment_end=6, state=shifted_state),
        SegmentInfo(segment_num=2, segment_start=7, segment_end=9, state=baseline_state),
    ],
    TimeseriesAnnotation(name="workflow_series"),
)
workflow_dataset = Dataset([workflow_provider])
workflow_bisegments = workflow_dataset.filter_by_bisegments()
workflow_selected = ColumnsSelectorTransformer(columns=["value"]).transform(workflow_provider)
print("Workflow provider change points:", list(workflow_provider.change_points))
print("Workflow dataset size:", len(workflow_dataset))
print("Workflow bisegment size:", len(workflow_bisegments))
print("Workflow selected columns:", list(workflow_selected.feature_columns))
display(workflow_selected.dataset().head())

### 9.2 Final summary

The PySATL data layer is not only a set of classes; it is a contract that keeps all later notebooks coherent. Raw providers solve storage and iteration, labeled providers solve ground-truth representation, datasets solve collection-level workflows, and transformers and loaders solve the practical edges where real data enters or changes shape.

That is why this notebook is so detailed. Everything later in the tutorial series assumes the reader can recognize these abstractions on sight and move between them confidently.
